In [1]:
from pathlib import Path
from typing import Optional

import joblib
import numpy as np
import pandas as pd

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize


# ============================================================
# CONFIG
# ============================================================

PROCESSED_DIR = Path("../../datasets/processed/PAN2011_300")
ARTIFACT_DIR = Path("../../artifacts/lsa")

SOURCE_CHUNKS_PATH = PROCESSED_DIR / "source_chunks_lsa_esa.parquet"
SUSPICIOUS_CHUNKS_PATH = PROCESSED_DIR / "suspicious_chunks_lsa_esa.parquet"

SUSPICIOUS_DOC_ID = "part1__suspicious-document00005.txt"

OUTPUT_PATH = PROCESSED_DIR / "lsa_candidates_suspicious_doc_00001.parquet"

# Change this to True only when you want to rebuild the source LSA index.
BUILD_INDEX = False


# ============================================================
# LOAD CHUNKS
# ============================================================

def load_lsa_esa_chunks(
    path: Path,
    text_column: str = "lsa_esa_text",
) -> pd.DataFrame:
    df = pd.read_parquet(path)

    required_columns = {
        "chunk_id",
        "doc_id",
        "chunk_index",
        "start_char",
        "end_char",
        text_column,
    }

    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns in {path}: {missing}")

    df = df.copy()
    df[text_column] = df[text_column].fillna("").astype(str)
    df = df[df[text_column].str.strip() != ""].reset_index(drop=True)

    return df


# ============================================================
# BUILD LSA INDEX
# ============================================================

def build_lsa_index(
    source_chunks_path: Path,
    artifact_dir: Path,
    text_column: str = "lsa_esa_text",
    n_components: int = 200,
    max_features: int = 100_000,
    max_source_chunks: Optional[int] = None,
):
    """
    Build and save the LSA source index.

    Creates:
    - tfidf_vectorizer.joblib
    - svd_model.joblib
    - source_lsa_vectors.npy
    - source_lsa_metadata.parquet
    """

    artifact_dir = Path(artifact_dir)
    artifact_dir.mkdir(parents=True, exist_ok=True)

    source_df = load_lsa_esa_chunks(
        path=source_chunks_path,
        text_column=text_column,
    )

    if max_source_chunks is not None:
        source_df = source_df.head(max_source_chunks).reset_index(drop=True)

    source_texts = source_df[text_column].tolist()

    print(f"Loaded {len(source_df)} source chunks")
    print("Fitting TF-IDF...")

    vectorizer = TfidfVectorizer(
        max_features=max_features,
        min_df=2,
        max_df=0.95,
        ngram_range=(1, 2),
        sublinear_tf=True,
        norm="l2",
    )

    vectorizer = TfidfVectorizer(
        max_features=max_features,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.85,
        sublinear_tf=True,
        lowercase=True,
        strip_accents="unicode",
        norm="l2",
    )

    source_tfidf = vectorizer.fit_transform(source_texts)

    print(f"TF-IDF matrix shape: {source_tfidf.shape}")

    feature_count = source_tfidf.shape[1]
    actual_components = min(n_components, feature_count - 1)

    if actual_components < 2:
        raise ValueError(
            f"Not enough TF-IDF features for LSA. Feature count: {feature_count}"
        )

    print(f"Fitting TruncatedSVD with {actual_components} components...")

    svd = TruncatedSVD(
        n_components=actual_components,
        random_state=42,
        n_iter=5,
    )

    source_lsa = svd.fit_transform(source_tfidf)

    print("Normalizing source LSA vectors...")
    source_lsa = normalize(source_lsa, norm="l2", axis=1)

    metadata_columns = [
        "chunk_id",
        "doc_id",
        "chunk_index",
        "start_char",
        "end_char",
    ]

    optional_columns = ["file_name", "relative_path", "part", "word_count"]
    metadata_columns += [col for col in optional_columns if col in source_df.columns]

    source_metadata = source_df[metadata_columns].copy()

    print("Saving LSA artifacts...")

    np.save(artifact_dir / "source_lsa_vectors.npy", source_lsa)
    source_metadata.to_parquet(
        artifact_dir / "source_lsa_metadata.parquet",
        index=False,
    )
    joblib.dump(svd, artifact_dir / "svd_model.joblib")
    joblib.dump(vectorizer, artifact_dir / "tfidf_vectorizer.joblib")

    print(f"Saved LSA artifacts to: {artifact_dir}")
    print(f"Explained variance ratio sum: {svd.explained_variance_ratio_.sum():.4f}")


# ============================================================
# LOAD LSA INDEX
# ============================================================

def load_lsa_index(artifact_dir: Path):
    artifact_dir = Path(artifact_dir)

    vectorizer_path = artifact_dir / "tfidf_vectorizer.joblib"
    svd_path = artifact_dir / "svd_model.joblib"
    vectors_path = artifact_dir / "source_lsa_vectors.npy"
    metadata_path = artifact_dir / "source_lsa_metadata.parquet"

    for path in [vectorizer_path, svd_path, vectors_path, metadata_path]:
        if not path.exists():
            raise FileNotFoundError(f"Missing LSA artifact: {path}")

    print("Loading LSA artifacts...")

    vectorizer = joblib.load(vectorizer_path)
    svd = joblib.load(svd_path)
    source_lsa = np.load(vectors_path)
    source_metadata = pd.read_parquet(metadata_path)

    return vectorizer, svd, source_lsa, source_metadata


# ============================================================
# QUERY ONE SUSPICIOUS DOCUMENT
# ============================================================

def retrieve_lsa_candidates_for_suspicious_doc(
    suspicious_chunks_path: Path,
    artifact_dir: Path,
    suspicious_doc_id: str,
    output_path: Path,
    text_column: str = "lsa_esa_text",
    top_k: int = 20,
    batch_size: int = 8,
    max_suspicious_chunks: Optional[int] = None,
) -> pd.DataFrame:
    """
    Query the LSA source index using only one selected suspicious document.

    Returns top-k source chunks for each suspicious chunk.
    """

    vectorizer, svd, source_lsa, source_metadata = load_lsa_index(artifact_dir)

    suspicious_df = load_lsa_esa_chunks(
        path=suspicious_chunks_path,
        text_column=text_column,
    )

    suspicious_df = suspicious_df[
        suspicious_df["doc_id"] == suspicious_doc_id
    ].copy()

    if suspicious_df.empty:
        raise ValueError(f"No suspicious chunks found for doc_id: {suspicious_doc_id}")

    if max_suspicious_chunks is not None:
        suspicious_df = suspicious_df.head(max_suspicious_chunks).reset_index(drop=True)

    print(f"Selected suspicious document: {suspicious_doc_id}")
    print(f"Suspicious chunks to query: {len(suspicious_df)}")
    print(f"Searching top-{top_k} source chunks per suspicious chunk")

    results = []

    for start in range(0, len(suspicious_df), batch_size):
        end = min(start + batch_size, len(suspicious_df))
        batch_df = suspicious_df.iloc[start:end]

        batch_texts = batch_df[text_column].tolist()

        suspicious_tfidf = vectorizer.transform(batch_texts)
        suspicious_lsa = svd.transform(suspicious_tfidf)
        suspicious_lsa = normalize(suspicious_lsa, norm="l2", axis=1)

        similarities = suspicious_lsa @ source_lsa.T

        for local_i, suspicious_row in enumerate(batch_df.itertuples(index=False)):
            sims = similarities[local_i]

            safe_top_k = min(top_k, len(sims))
            top_indices = np.argpartition(-sims, safe_top_k - 1)[:safe_top_k]
            top_indices = top_indices[np.argsort(-sims[top_indices])]

            for rank, source_idx in enumerate(top_indices, start=1):
                source_row = source_metadata.iloc[source_idx]

                results.append({
                    "suspicious_chunk_id": suspicious_row.chunk_id,
                    "suspicious_doc_id": suspicious_row.doc_id,
                    "suspicious_chunk_index": suspicious_row.chunk_index,
                    "suspicious_start_char": suspicious_row.start_char,
                    "suspicious_end_char": suspicious_row.end_char,

                    "source_chunk_id": source_row["chunk_id"],
                    "source_doc_id": source_row["doc_id"],
                    "source_chunk_index": source_row["chunk_index"],
                    "source_start_char": source_row["start_char"],
                    "source_end_char": source_row["end_char"],

                    "LSA_score": float(sims[source_idx]),
                    "LSA_rank": rank,
                })

        print(f"Processed suspicious chunks {start} to {end}")

    output_df = pd.DataFrame(results)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_df.to_parquet(output_path, index=False)

    print(f"Saved LSA candidates to: {output_path}")

    return output_df


# ============================================================
# DOCUMENT-LEVEL MEAN SCORE
# ============================================================

def get_top_source_documents_by_mean_score(
    candidates_df: pd.DataFrame,
    source_chunks_path: Path,
    suspicious_doc_id: str,
    top_n: int = 20,
    min_match_count: int = 4,
    output_path: Optional[Path] = None,
) -> pd.DataFrame:
    """
    Rank unique source documents by mean LSA score.

    If the same source document appears multiple times because of different
    source chunks, this computes the mean score for that source document.
    """

    required_columns = {
        "suspicious_doc_id",
        "suspicious_chunk_id",
        "source_doc_id",
        "source_chunk_id",
        "LSA_score",
    }

    missing = required_columns - set(candidates_df.columns)
    if missing:
        raise ValueError(f"Missing columns in candidates_df: {missing}")

    filtered_df = candidates_df[
        candidates_df["suspicious_doc_id"] == suspicious_doc_id
    ].copy()

    if filtered_df.empty:
        raise ValueError(f"No candidates found for suspicious_doc_id: {suspicious_doc_id}")

    grouped_df = (
        filtered_df
        .groupby("source_doc_id")
        .agg(
            mean_LSA_score=("LSA_score", "mean"),
            max_LSA_score=("LSA_score", "max"),
            min_LSA_score=("LSA_score", "min"),
            match_count=("LSA_score", "count"),
            unique_source_chunks=("source_chunk_id", "nunique"),
            unique_suspicious_chunks=("suspicious_chunk_id", "nunique"),
        )
        .reset_index()
    )

    grouped_df = grouped_df[grouped_df["match_count"] >= min_match_count].copy()

    grouped_df = (
        grouped_df
        .sort_values(["mean_LSA_score", "match_count"], ascending=[False, False])
        .head(top_n)
        .reset_index(drop=True)
    )

    grouped_df["source_doc_rank"] = range(1, len(grouped_df) + 1)

    grouped_df = grouped_df[
        [
            "source_doc_rank",
            "source_doc_id",
            "mean_LSA_score",
            "max_LSA_score",
            "min_LSA_score",
            "match_count",
            "unique_source_chunks",
            "unique_suspicious_chunks",
        ]
    ]

    source_meta_df = pd.read_parquet(
        source_chunks_path,
        columns=["doc_id", "relative_path"],
    ).drop_duplicates("doc_id")

    source_meta_df = source_meta_df.rename(columns={
        "doc_id": "source_doc_id",
        "relative_path": "source_relative_path",
    })

    grouped_df = grouped_df.merge(
        source_meta_df,
        on="source_doc_id",
        how="left",
    )

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        grouped_df.to_parquet(output_path, index=False)
        print(f"Saved top source documents to: {output_path}")

    return grouped_df

def get_top_source_documents_by_max_score(
    candidates_df: pd.DataFrame,
    source_chunks_path: Path,
    suspicious_doc_id: str,
    top_n: int = 20,
    min_match_count: int = 1,
    output_path: Optional[Path] = None,
) -> pd.DataFrame:
    """
    Rank unique source documents by strongest LSA chunk match.

    This is better for plagiarism retrieval because plagiarism is often local:
    one strong matching passage can identify the true source.
    """

    required_columns = {
        "suspicious_doc_id",
        "suspicious_chunk_id",
        "source_doc_id",
        "source_chunk_id",
        "LSA_score",
    }

    missing = required_columns - set(candidates_df.columns)
    if missing:
        raise ValueError(f"Missing columns in candidates_df: {missing}")

    filtered_df = candidates_df[
        candidates_df["suspicious_doc_id"] == suspicious_doc_id
    ].copy()

    if filtered_df.empty:
        raise ValueError(f"No candidates found for suspicious_doc_id: {suspicious_doc_id}")

    grouped_df = (
        filtered_df
        .groupby("source_doc_id")
        .agg(
            mean_LSA_score=("LSA_score", "mean"),
            max_LSA_score=("LSA_score", "max"),
            min_LSA_score=("LSA_score", "min"),
            match_count=("LSA_score", "count"),
            unique_source_chunks=("source_chunk_id", "nunique"),
            unique_suspicious_chunks=("suspicious_chunk_id", "nunique"),
        )
        .reset_index()
    )

    grouped_df = grouped_df[grouped_df["match_count"] >= min_match_count].copy()

    grouped_df = (
        grouped_df
        .sort_values(
            ["max_LSA_score", "match_count", "mean_LSA_score"],
            ascending=[False, False, False],
        )
        .head(top_n)
        .reset_index(drop=True)
    )

    grouped_df["source_doc_rank"] = range(1, len(grouped_df) + 1)

    grouped_df = grouped_df[
        [
            "source_doc_rank",
            "source_doc_id",
            "mean_LSA_score",
            "max_LSA_score",
            "min_LSA_score",
            "match_count",
            "unique_source_chunks",
            "unique_suspicious_chunks",
        ]
    ]

    source_meta_df = pd.read_parquet(
        source_chunks_path,
        columns=["doc_id", "relative_path"],
    ).drop_duplicates("doc_id")

    source_meta_df = source_meta_df.rename(columns={
        "doc_id": "source_doc_id",
        "relative_path": "source_relative_path",
    })

    grouped_df = grouped_df.merge(
        source_meta_df,
        on="source_doc_id",
        how="left",
    )

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        grouped_df.to_parquet(output_path, index=False)
        print(f"Saved top source documents to: {output_path}")

    return grouped_df


# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    BUILD_INDEX=False
    SUSPICIOUS_DOC_ID = "part14__suspicious-document06510.txt"
    if BUILD_INDEX:
        build_lsa_index(
            source_chunks_path=SOURCE_CHUNKS_PATH,
            artifact_dir=ARTIFACT_DIR,
            n_components=200,
            max_features=100000,
            max_source_chunks=None,
        )

    candidates_df = retrieve_lsa_candidates_for_suspicious_doc(
        suspicious_chunks_path=SUSPICIOUS_CHUNKS_PATH,
        artifact_dir=ARTIFACT_DIR,
        suspicious_doc_id=SUSPICIOUS_DOC_ID,
        output_path=OUTPUT_PATH,
        top_k=500,              
        batch_size=8,
        max_suspicious_chunks=None,
    )

    top_sources_df = get_top_source_documents_by_max_score(
        candidates_df=candidates_df,
        source_chunks_path=PROCESSED_DIR / "source_chunks.parquet",
        suspicious_doc_id=SUSPICIOUS_DOC_ID,
        top_n=50,              
        min_match_count=1,     
        output_path=PROCESSED_DIR / "lsa_top_source_documents_by_max_score.parquet",
    )


print("\nTOP SOURCE DOCUMENTS BY MEAN LSA SCORE")
top_sources_df

Loading LSA artifacts...
Selected suspicious document: part14__suspicious-document06510.txt
Suspicious chunks to query: 303
Searching top-500 source chunks per suspicious chunk
Processed suspicious chunks 0 to 8
Processed suspicious chunks 8 to 16
Processed suspicious chunks 16 to 24
Processed suspicious chunks 24 to 32
Processed suspicious chunks 32 to 40
Processed suspicious chunks 40 to 48
Processed suspicious chunks 48 to 56
Processed suspicious chunks 56 to 64
Processed suspicious chunks 64 to 72
Processed suspicious chunks 72 to 80
Processed suspicious chunks 80 to 88
Processed suspicious chunks 88 to 96
Processed suspicious chunks 96 to 104
Processed suspicious chunks 104 to 112
Processed suspicious chunks 112 to 120
Processed suspicious chunks 120 to 128
Processed suspicious chunks 128 to 136
Processed suspicious chunks 136 to 144
Processed suspicious chunks 144 to 152
Processed suspicious chunks 152 to 160
Processed suspicious chunks 160 to 168
Processed suspicious chunks 168 

,source_doc_rank,source_doc_id,mean_LSA_score,max_LSA_score,min_LSA_score,match_count,unique_source_chunks,unique_suspicious_chunks,source_relative_path
0,1,part14__source-document06533.txt,0.598984,0.836219,0.498086,1916,314,195,part14/source-document06533.txt
1,2,part22__source-document10911.txt,0.613269,0.763828,0.509693,53,33,27,part22/source-document10911.txt
2,3,part4__source-document01971.txt,0.598801,0.753782,0.530075,585,121,112,part4/source-document01971.txt
3,4,part14__source-document06666.txt,0.607028,0.751409,0.530253,67,48,29,part14/source-document06666.txt
4,5,part14__source-document06730.txt,0.597954,0.746429,0.513050,513,78,84,part14/source-document06730.txt
5,6,part20__source-document09614.txt,0.593349,0.746245,0.522785,65,47,34,part20/source-document09614.txt
6,7,part14__source-document06900.txt,0.590728,0.734258,0.521439,46,37,35,part14/source-document06900.txt
7,8,part6__source-document02966.txt,0.603952,0.734165,0.519058,126,94,40,part6/source-document02966.txt
8,9,part14__source-document06882.txt,0.598012,0.733573,0.536004,151,33,46,part14/source-document06882.txt
9,10,part1__source-document00154.txt,0.600369,0.732659,0.522905,44,34,25,part1/source-document00154.txt
